# Sliding Window Maximum (1D Kernel Max)

[Back to Session Overview](../README.md) · [Back to Module 01](../../README.md)

Sliding window operations are foundational across streaming telemetry, computer vision convolution kernels, and sequential model architectures. This notebook explores the 1D Kernel Maximum problem across brute force slicing, anchor-radius indexing, boundary-aware padding, and the optimal $O(N)$ monotonic double-ended queue.

### Objectives
1. Implement the classic sliding window maximum using sub-array slicing and queue simulation.
2. Formulate symmetric anchor-radius kernel indexing.
3. Handle boundary conditions using truncated windows and edge padding.
4. Implement the optimal $O(N)$ Monotonic Deque algorithm and benchmark its efficiency.


## 1. Problem Formulation and Brute Force Slicing

Given an array of numbers `num_list` of length $N$ and a window size $k$ ($1 \le k \le N$), compute the maximum value for each contiguous subarray of size $k$ moving from left to right.
The output array has length $N - k + 1$.


In [ ]:
def sliding_window_max_slicing(num_list: list[int | float], k: int) -> list[int | float]:
    """Compute sliding window maximum using direct sub-list slicing."""
    if not num_list or k <= 0:
        return []
    if k > len(num_list):
        return [max(num_list)]

    n = len(num_list)
    result = []
    for i in range(n - k + 1):
        window = num_list[i : i + k]
        result.append(max(window))
    return result


# Verification on slide example
example_input = [3, 4, 5, 1, -44, 5, 10, 12, 33, 1]
k_val = 3
expected_out = [5, 5, 5, 5, 10, 12, 33, 33]
res_slice = sliding_window_max_slicing(example_input, k_val)
print("Slicing output:", res_slice)
assert res_slice == expected_out

## 2. Anchor-Radius Kernel Formulation & Boundary Awareness

In signal and image processing, kernels are frequently centered on an anchor point $i$, with a radius $r = \lfloor k / 2 \rfloor$.
For odd kernel sizes $k = 2r + 1$, interior anchors run from $r$ to $N - 1 - r$.
Near sequence boundaries ($i < r$ or $i \ge N - r$), two strategies exist:
1. **Clamped / Truncated Windows:** Shrink window boundaries to $[\max(0, i - r), \min(N, i + r + 1)]$.
2. **Replication / Constant Padding:** Pad the boundaries with edge values before applying standard kernel scans.


In [ ]:
def sliding_max_boundary_aware(num_list: list[int | float], k: int) -> list[int | float]:
    """Anchor-centered sliding maximum with boundary clamping."""
    if not num_list or k <= 0:
        return []

    n = len(num_list)
    radius = k // 2
    result = []

    for i in range(n):
        begin = max(0, i - radius)
        end = min(n, i + radius + 1)
        result.append(max(num_list[begin:end]))

    return result


demo_signal = [3, 4, 6, 2, -1, 5]
clamped_k3 = sliding_max_boundary_aware(demo_signal, k=3)
print("Original signal:        ", demo_signal)
print("Boundary-aware (k=3) max:", clamped_k3)

# Anchor 0: window [3, 4] -> max 4
# Anchor 1: window [3, 4, 6] -> max 6
# Anchor 2: window [4, 6, 2] -> max 6
# Anchor 3: window [6, 2, -1] -> max 6
# Anchor 4: window [2, -1, 5] -> max 5
# Anchor 5: window [-1, 5] -> max 5
assert clamped_k3 == [4, 6, 6, 6, 5, 5]

## 3. Optimal $O(N)$ Monotonic Double-Ended Queue

While brute-force slicing requires $O(N \cdot k)$ operations, the optimal algorithmic solution achieves linear $O(N)$ time using a **Monotonic Decreasing Deque**.
- We store indices in the deque.
- Elements in the deque are maintained in strictly decreasing order of their values.
- For each new element:
  1. Evict indices outside the current window $[i - k + 1, i]$ from the front.
  2. Evict indices whose values are less than or equal to `num_list[i]` from the back.
  3. Push current index $i$ to the back.
  4. The front of the deque always points to the maximum element for window ending at $i$.


In [ ]:
from collections import deque


def sliding_window_max_monotonic(num_list: list[int | float], k: int) -> list[int | float]:
    """Compute sliding window maximum in optimal O(N) time via monotonic deque."""
    if not num_list or k <= 0:
        return []
    if k > len(num_list):
        return [max(num_list)]

    dq: deque[int] = deque()  # stores indices
    result: list[int | float] = []

    for i, val in enumerate(num_list):
        # 1. Evict elements outside the current window
        if dq and dq[0] < i - k + 1:
            dq.popleft()

        # 2. Maintain monotonic decreasing property
        while dq and num_list[dq[-1]] <= val:
            dq.pop()

        # 3. Add current element index
        dq.append(i)

        # 4. Record maximum once the first full window is reached
        if i >= k - 1:
            result.append(num_list[dq[0]])

    return result


res_mono = sliding_window_max_monotonic(example_input, k_val)
print("Monotonic deque output:", res_mono)
assert res_mono == expected_out

In [ ]:
import csv
from pathlib import Path

# Verification on real sensor telemetry
sensor_csv = Path("../data/sensor_stream.csv")
temps = []
with sensor_csv.open(encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        temps.append(float(row["temperature_celsius"]))

max_temps_slice = sliding_window_max_slicing(temps, k=5)
max_temps_mono = sliding_window_max_monotonic(temps, k=5)

assert len(max_temps_slice) == len(temps) - 5 + 1
assert max_temps_slice == max_temps_mono
print(f"Verified {len(max_temps_mono)} window maxima on sensor stream.")